# Note to Marker
The python kernel used for this project is Python 3.12.12

Citations and references appear in the cells of reference


# 1 - Domain Specific area and objectives of the project (200 - 500 words)

My domain interest is in the circular economy, specifically the recovery of precious metals from electronic waste.  A major challenge in any recycling or circular economy project is the act of sorting (90% of the recovery effort based on my personal experience).  Scrap electronics circuit boards contain a mixture of components, materials and value, and these need to be separated before precious metal recovery can occur. Sorting can be performed manually, mechanically, by hydro separation, or robotic methods. My particular interest is in robotic sorting, where a robot arm identifies (computer vision), picks, separates and sorts electronic components as one of the initial steps for precious metal recovery.

With this domain in mind, I began a search for a dataset that met the project requirements.  Although I found many electronics and robotics datasets, many were aimed at image classification or object recognition (precisely 'my domain') rather than linear regression.  The dataset that I have chosen is a UC Irvine dataset: [Condition Monitoring of Hydraulic Systems](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems). [1] This dataset is not directly related to e-waste sorting, but it is closely related to robotic automations and control. Robotic and automated sorting systems depend on reliable motion (360,x,y,z axis), actuation, cooling and mechanical control. The device measured in the UC Irvine study was a hydraulic test rig and is a useful proxy for understanding how sensor data can be used to predict machine performance.

The objective of this project is to use Linear Regression to predict a continuous hydraulic system performance measure such as cooling efficiency or 'systems efficiency factor', from sensor readings including pressure, flow, temperature, vibration and motor power.  A by-product of measuring  these variables is the identification of faulty, or close to failure sensors, so that they can be maintained or replaced.  Improving efficiency may also lead to an improved ROI and add to an organizations bottom line through the use of inferred maintenance leading to improved uptime and system performance.

Linear Regression is suitable because the dataset contains numerical sensor values and numerical target values, and the project requirements require identifying variables with an approximately linear trendline. I will use visualization techniques to inspect the dataset such as scatter plots, correlelations and trendlines, before selecting predictors.

The project will preprocess the raw data (multiple files) into cycle level features, normalize values, and train a Linear Regression model with a subsequent evaluation for accuracy and error.  The intention is to understand how the model can identify how sensor readings are associated with hydraulic effienciency.  

In the wider circular economy context, this supports the challenge of robotic reliability.  Many of these systems run 24 hours a day with millions of movements so identifying loss of efficiency, prediction of maintenance or component replacement, contributes to improved uptime and to overall efficiency of recovery operations.

## Goal:
### To predict hydraulic system efficiency from sensor data to support robotic sorting in e-waste precious metal recover

[1]: Helwig,N., et al. (2018) UC Irvine. Last accessed June 21, 2026. "Condition Monitoring of Hydraulic Systems"


# 2 - Dataset Description (200-500 words)
The dataset selected for this project is the UC Irvine [Condition Monitoring of Hydraulic Systems](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems). [1] created by Helwig, Pignanelli, and Schütze and donated to the UCI Machine Learning Repository [2] in 2018.  The metadata associated with the repository states that 'the dataset was experimentally obtained with a hydraulic test rig' that was subjectd to a "load cycle of 60 seconds where measurements of pressure, volume flows and temperatures were taken while the hydraulic components (cooler, valve, pump and actuator) were quantitatively varied"[1].

The dataset is licenced under a [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/legalcode)[3] which allows for the sharing and adaptation of the dataset for any purpose, provided that the appropriate credit is given.

The dataset contains 2,205 instance where each instance is one 60 second load cycle. The number of features is 43,680 based on 14 sensors, and is significant due to their sheer number of features.  When I understood the challenge of working with this number of attributes I felt anxious but decided to remain with the dataset and the challenges that it presents.  The dataset comprises of multiple tab-delimited text files where each file represents a single sensor. 

The file structure is a matrix style where a row represents a 60 second cycle, and the columns represent data points. The immediate challenge will be to merge these into a single file (DataFrame) and then figure out how to simplify and normalize the resulting data.

The variables (attributes/features) represent pressure sensors, volume flow sensors, temperature sensors, vibration and motor power. Targets include cooling efficiency, cooling power, and system efficiency factor. The dataset is continuous and numeric with a combination of numbers and percentages.

The 14 sensors are identified by the name of their data files and there are 5 groupings:

- Pressure Sensors PS1 to PS6.  Measured in bars at 100Hz 
- Motor Power EPS1. Measured in Watts at 100hz
- Volume Flow FS1, FS2. Measured in liters per minute at 100Hz
- Temperature TS1 to TS4. Measured in °C at 1Hz
- Vibration VS1. 

In terms of missing values the dataset metadata states that there are no missing values [1]

The dataset is suitable for a Linear Regression project (it is listed as such [1]) and there are three possible targets: (1) hydraulic system efficiency, (2) cooling efficiency or (3) cooling power. A Linear Regression model could test whether a change in sensor readings correlate with system performance in a linear fashion.  

[1]: Helwig,N., et al. (2018) UC Irvine. Last accessed June 21, 2026. "Condition Monitoring of Hydraulic Systems". https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems
[2]: UC Irvine. Machine Learning Repository.  Last accessed June 21, 2026. https://archive.ics.uci.edu/
[3]: Creative Commons. CC By 4.0 Attribution 4.0 International Legal Code. Last accessed June 21, 2026. https://creativecommons.org/licenses/by/4.0/legalcode



# 3. Data Preparation
This is a large section.  Primarily due to the number of files that are needed to be processed as well as both visual and programmatic analysis of the data.

The input consists of 17 tab delimited data files, a profile file, and 2 metadata files (description and documentation).
Each type of file will be progressively analysed and explained as this section progresses.

The output will be a sensor dataframe containing statistical measures for each sensor cycle that can be used for machine learning purposes in later sections of the project.

There will be some data visualization appearing in this section as I progress through the process of file combination and normalization.  

In [21]:
# Import libraries
import numpy as np
import pandas as pd

from pathlib import Path
import zipfile

In [22]:
# Display settings
# These are my own personal preferences
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

The original dataset is downloaded as a zip file and stored in the /data/original directory.
Doing this means that the original zip can be submitted along with the project and then
extracted using the next few steps

The original zip download uses a '+' delimiting each word.  When I downloaded I renamed the file

from: condition+monitoring+of+hydraulic+systems.zip

to: condition_monitoring_of_hydraulic-systems.zip

In [53]:
# Set the data source paths
project_dir = Path.cwd()
# print(project_dir)
data_dir = project_dir / "data" / "extracted"
zip_dir = project_dir / "data" / "original"
dataframe_dir = project_dir / "data" / "dataframes"
original_path = zip_dir / "condition_monitoring_of_hydraulic-systems.zip"
extraction_path = data_dir / "condition_monitoring_of_hydraulic-systems"

The dataset is distributed as a ZIP archive. After extraction, the folder contains multiple tab-delimited text files. The data is not supplied as a single tidy CSV file, so loading and combining the files is part of the preprocessing task.

In [45]:
# delete any files that exist in the extraction_path prior to extraction
for item in extraction_path.iterdir():
    item.unlink()

In [46]:
# Extract the zip file
with zipfile.ZipFile(original_path, "r") as zip_ref:
    zip_ref.extractall(extraction_path)

In my experience it is best to step through each and every step and automate as much as possible.

I will first create a dataframe to store information about each file extracted including:

- filename
- relative file path (important per UoL instructions to use relative paths)
- file size (mb)
- file extension (should be .txt)

In [47]:
# recursive search for all (.*) files using rglob
all_data_files = sorted(extraction_path.rglob("*"))

data_file_inventory = [] # initialise an empty python list to store files

In [48]:
# Add dict entries containing file metadata for each extracted file to the data_file_inventory_list
for file in all_data_files:
    if file.is_file():
        data_file_inventory.append({
            "file_name": file.name,
            "relative_path": str(file.relative_to(extraction_path)),
            "extension": file.suffix,
            "size": file.stat().st_size / (1024 * 1024) # convert file size to mb
        })

In [49]:
# convert the data_file_inventory_list to a pandas dataframe
data_file_inventory_df = pd.DataFrame(data_file_inventory)
data_file_inventory_df

,file_name,relative_path,extension,size
0,CE.txt,CE.txt,.txt,0.869289
1,CP.txt,CP.txt,.txt,0.743239
2,description.txt,description.txt,.txt,0.003857
3,documentation.txt,documentation.txt,.txt,0.004500
4,EPS1.txt,EPS1.txt,.txt,83.362192
5,FS1.txt,FS1.txt,.txt,7.230688
6,FS2.txt,FS2.txt,.txt,7.888721
7,profile.txt,profile.txt,.txt,0.029751
8,PS1.txt,PS1.txt,.txt,87.158100
9,PS2.txt,PS2.txt,.txt,78.557597


The dataset is a combination of sensor reading files and additional files that describe the dataset.
The data for these files came directly from the dataset page on the UC Irvine site:
[variable information](https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems)

- Pressure files are prefixed with 'PS', unit is bars, sampling rate is 100Hz
- Motor Power files are prefixed with 'EPS'
- Volume flow files are prefixed with 'FS'
- Temperature files are prefixed with 'TS'
- Vibration files are prefixed with 'VS'
- Cooling Efficiency filesare prefixed with 'CE'
- Cooling Power files are prefixed with 'CP'
- Efficiency Factor files are prefixed with 'SE'  #TODO: is this the target?

Each file has rows that represent a 60 second cycle.  For each cycle, readings are taken based on the sampling rate. So, a 100 Hz sampling rate is 100 readings per second.  For the Pressure Sensor "PS1" file this means that that there are 2,205 cycles each with 100 samples per second for 60 seconds: 100 * 60 = 6,000 samples per cycle.

This equates to 2,205 * 6,000 = 132,300 samples

The profile.txt file has the associated 'machine conditions' for each cycle. The profile.txt file 'could' be used as a taeget file to understand, for example, if the combination of pressure, flow, temperature, vibration and motor power readings could be used to predicy cooler efficiency.  At this stage, I do not know enough about the dataset to explore this question but it is something that will be reviewed once I begin asking some questions of the datasets themselves.

profile.txt : TODO: Add more detail
- Column 0: Cooler Condition.
- Column 1: Valve Condition.
- Column 2: Internal Pump Leakage. 
- Column 3: Hydraulic Accumulator Pressure
- Column 4: Stable Flag (0 or 1).


In [50]:
# I needed a way to prevent metadata files being loaded so I created a dictionary
# to store a list of files that I can load into a single dataframe
sensor_metadata = {
    "PS1": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "PS2": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "PS3": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "PS4": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "PS5": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "PS6": {"quantity": "Pressure", "unit": "bar", "sampling_rate_hz": 100},
    "EPS1": {"quantity": "Motor power", "unit": "W", "sampling_rate_hz": 100},
    "FS1": {"quantity": "Volume flow", "unit": "l/min", "sampling_rate_hz": 10},
    "FS2": {"quantity": "Volume flow", "unit": "l/min", "sampling_rate_hz": 10},
    "TS1": {"quantity": "Temperature", "unit": "C", "sampling_rate_hz": 1},
    "TS2": {"quantity": "Temperature", "unit": "C", "sampling_rate_hz": 1},
    "TS3": {"quantity": "Temperature", "unit": "C", "sampling_rate_hz": 1},
    "TS4": {"quantity": "Temperature", "unit": "C", "sampling_rate_hz": 1},
    "VS1": {"quantity": "Vibration", "unit": "mm/s", "sampling_rate_hz": 1},
    "CE": {"quantity": "Cooling efficiency", "unit": "%", "sampling_rate_hz": 1},
    "CP": {"quantity": "Cooling power", "unit": "kW", "sampling_rate_hz": 1},
    "SE": {"quantity": "Efficiency factor", "unit": "%", "sampling_rate_hz": 1},
}

Create a loading summary to record the number of rows, columns, total cells and missing values for each file. This confirms whether the files have been read correctly and whether they align by operating cycle.

In [ ]:
# Load the datasets (about 10 seconds)

txt_files = sorted(extraction_path.glob("*.txt"))

data_files = {}
load_summary = []

for file in txt_files:
    name = file.stem
    try:
        df = pd.read_csv(file, sep="\t", header=None)
        data_files[name] = df

        load_summary.append({
            "file": file.name,
            "key": name,
            "rows" : df.shape[0],
            "columns": df.shape[1],
            "total_cells": df.shape[0] * df.shape[1],
            "missing_values": int(df.isna().sum().sum()),
            "numeric_columns": int(df.select_dtypes(include=[np.number]).shape[1])
        })

    except Exception as e:
       load_summary.append({
            "file": file.name,
            "key": name,
            "rows" : None,
            "columns": None,
            "total_cells": None,
            "missing_values": None,
            "numeric_columns": None,
            "error": str(e)
        }) 
       
load_summary_df = pd.DataFrame(load_summary)
load_summary_df

,file,key,rows,columns,total_cells,missing_values,numeric_columns,error
0,CE.txt,CE,2205.0,60.0,132300.0,0.0,60.0,NaN
1,CP.txt,CP,2205.0,60.0,132300.0,0.0,60.0,NaN
2,description.txt,description,NaN,NaN,NaN,NaN,NaN,'utf-8' codec can't decode byte 0xfc in positi...
3,documentation.txt,documentation,NaN,NaN,NaN,NaN,NaN,'utf-8' codec can't decode byte 0xfc in positi...
4,EPS1.txt,EPS1,2205.0,6000.0,13230000.0,0.0,6000.0,NaN
5,FS1.txt,FS1,2205.0,600.0,1323000.0,0.0,600.0,NaN
6,FS2.txt,FS2,2205.0,600.0,1323000.0,0.0,600.0,NaN
7,profile.txt,profile,2205.0,5.0,11025.0,0.0,5.0,NaN
8,PS1.txt,PS1,2205.0,6000.0,13230000.0,0.0,6000.0,NaN
9,PS2.txt,PS2,2205.0,6000.0,13230000.0,0.0,6000.0,NaN


In [54]:
# Save the dataframe
load_summary_df.to_csv(dataframe_dir / "load_summary.csv", index = False)

The profile file is treated differently from the raw sensor files because it contains one row per operating cycle rather than in-cycle sampled sensor readings. 

These columns describe the known condition of hydraulic components and can be used for grouping, filtering or later classification tasks. They can also help interpret whether efficiency changes are associated with degraded component states.

In [55]:
# process the profile file.
profile = data_files["profile"].copy()

profile.columns=[
    "cooler_condition",
    "valve_condition",
    "pump_leakage",
    "hydraulic_accumulator",
    "stable_flag"
]

profile.head()

,cooler_condition,valve_condition,pump_leakage,hydraulic_accumulator,stable_flag
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


In [ ]:
# use describe to get see some file statistics
profile.describe()

,cooler_condition,valve_condition,pump_leakage,hydraulic_accumulator,stable_flag
count,2205.000000,2205.000000,2205.000000,2205.000000,2205.000000
mean,41.240816,90.693878,0.669388,107.199546,0.342857
std,42.383143,10.681802,0.817233,16.435848,0.474772
min,3.000000,73.000000,0.000000,90.000000,0.000000
25%,3.000000,80.000000,0.000000,90.000000,0.000000
50%,20.000000,100.000000,0.000000,100.000000,0.000000
75%,100.000000,100.000000,1.000000,130.000000,1.000000
max,100.000000,100.000000,2.000000,130.000000,1.000000


In [57]:
for col in profile.columns:
    print("\n", col)
    print(profile[col].value_counts().sort_index())


 cooler_condition
cooler_condition
3      732
20     732
100    741
Name: count, dtype: int64

 valve_condition
valve_condition
73      360
80      360
90      360
100    1125
Name: count, dtype: int64

 pump_leakage
pump_leakage
0    1221
1     492
2     492
Name: count, dtype: int64

 hydraulic_accumulator
hydraulic_accumulator
90     808
100    399
115    399
130    599
Name: count, dtype: int64

 stable_flag
stable_flag
0    1449
1     756
Name: count, dtype: int64


The results above are interesting.  Each sensor profile produces a small number of values (3 for example in the cooler_condition profile).  At this point in time I do not fully understand the implications of each result and will hopefully obtain a decent understanding as I work through the datasets.

The row alignment check (cell below) verifies that each file contains the same number of operating cycles. 

Important because later preprocessing will combine cycle-level summaries from each sensor file into one modelling table.

In [58]:
# check that every file has the same number of rows (per sensor) as found in profile.txt.
# This ensures a one to one mapping between sensor cycles and machine assessment found in profile.txt
row_check = []
expected_rows = profile.shape[0]

for name, df in data_files.items():
    row_check.append({
        "file_key": name,
        "rows": df.shape[0],
        "matches_profile_rows" : df.shape[0] == expected_rows
    })

row_check_df = pd.DataFrame(row_check)
row_check_df

,file_key,rows,matches_profile_rows
0,CE,2205,True
1,CP,2205,True
2,EPS1,2205,True
3,FS1,2205,True
4,FS2,2205,True
5,profile,2205,True
6,PS1,2205,True
7,PS2,2205,True
8,PS3,2205,True
9,PS4,2205,True


In [59]:
# save the row_check_df
row_check_df.to_csv(dataframe_dir / "row_check", index=False)

## Section 3 - Data Preparation - Programmatic data checks

Data Quality

The quality summary checks whether each file is numeric, whether missing values are present, and whether the range of values appears reasonable. Although UCI identifies the dataset as having no missing values, this check is still necessary because imported files may contain parsing issues, blank columns or formatting problems.

In [61]:
data_quality_summary = []

for name, df in data_files.items():
    # Ensure all number fields are either numeric or NaN
    numeric_df = df.apply(pd.to_numeric, errors="coerce")

    # convert to a numpy array (improved performance) and flatten to a list using ravel
    values = numeric_df.to_numpy().ravel()
    # remove NaNs and keep the array clean and numeric only
    values = values[~np.isnan(values)]

    data_quality_summary.append({
        "file_key" : name, 
        "rows" : df.shape[0],
        "columns" : df.shape[1],
        "missing_values": int(numeric_df.isna().sum().sum()),
        "min": np.min(values) if len(values) > 0 else np.nan,
        "max": np.max(values) if len(values) > 0 else np.nan,
        "mean": np.mean(values) if len(values) > 0 else np.nan,
        "std": np.std(values) if len(values) > 0 else np.nan,
    })

data_quality_summary_df = pd.DataFrame(data_quality_summary)
data_quality_summary_df

,file_key,rows,columns,missing_values,min,max,mean,std
0,CE,2205,60,0,17.042,48.777,31.299077,11.577906
1,CP,2205,60,0,1.016,2.909,1.808399,0.279326
2,EPS1,2205,6000,0,2097.800,2995.200,2495.509203,218.222229
3,FS1,2205,600,0,0.000,20.479,6.198549,3.213826
4,FS2,2205,600,0,8.764,10.453,9.649453,0.449489
5,profile,2205,5,0,0.000,130.000,48.029297,49.122090
6,PS1,2205,6000,0,133.130,191.920,160.485315,16.133330
7,PS2,2205,6000,0,0.000,167.770,109.379906,48.103172
8,PS3,2205,6000,0,0.000,18.828,1.753227,0.934707
9,PS4,2205,6000,0,0.000,10.266,2.600266,4.297607


The above dataframe is a good result.  As per the UC Irvine documentation there are no missing values which indicates that this is a very good quality dataset.


In [62]:
# save the dataframe
data_quality_summary_df.to_csv(dataframe_dir / "data_quality_summary.csv", index = False)

In [ ]:
# at 10